# Práctica 1 - Visualización de datos

## Objetivo
Analizar las temperaturas máximas y mínimas registradas cerca de Ann Arbor, Michigan, usando Pandas y Matplotlib. Se comparan los récords diarios del período 2005–2014 con las observaciones de 2015.

El archivo contiene una observación por fila: `id` identifica la estación, `date` es la fecha, `element` indica `TMAX` (máxima) o `TMIN` (mínima), y `value` es la temperatura en décimas de °C.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Cargamos el conjunto de datos incluido en la carpeta data.
ruta = '../data/fb441e62df2d58994928907a91895ec62c2c42e6cd075c2700843b89.csv'
df = pd.read_csv(ruta)

# Normalizamos los nombres de las columnas para trabajar con nombres consistentes.
df.columns = df.columns.str.lower().str.strip()
# En algunas versiones del dataset, la temperatura se llama data_value.
if 'data_value' in df.columns and 'value' not in df.columns:
    df = df.rename(columns={'data_value': 'value'})
df['date'] = pd.to_datetime(df['date'])

# Convertimos las temperaturas de décimas de grado a grados Celsius.
df['temperature_c'] = df['value'] / 10

# Revisamos las primeras filas y la estructura del DataFrame.
display(df.head())
print('Cantidad de filas y columnas:', df.shape)
print('\nColumnas y tipos de datos:')
df.info()
print('\nTipos de medición:')
display(df['element'].value_counts())


## 1. Preparar los datos
Para comparar el mismo día del año entre diferentes años, creamos una clave con el mes y el día. Quitamos el 29 de febrero, tal como permite la consigna, para que los años bisiestos no afecten la comparación. Los valores se mantienen en grados Celsius.

In [ ]:
# Excluimos el 29 de febrero.
df = df[~((df['date'].dt.month == 2) & (df['date'].dt.day == 29))].copy()

# Separamos el período de referencia y el año que queremos comparar.
periodo_referencia = df[df['date'].dt.year.between(2005, 2014)].copy()
datos_2015 = df[df['date'].dt.year == 2015].copy()

# La clave mes-día permite alinear, por ejemplo, todos los 1 de enero.
for datos in (periodo_referencia, datos_2015):
    datos['month_day'] = datos['date'].dt.strftime('%m-%d')

print('Observaciones 2005–2014:', len(periodo_referencia))
print('Observaciones 2015:', len(datos_2015))
print('Observaciones de 2015:', datos_2015['date'].dt.year.unique())

## 2. Calcular los récords diarios de 2005–2014
Para cada día del año, calculamos la temperatura máxima más alta (`TMAX`) y la temperatura mínima más baja (`TMIN`) del período de referencia.

In [ ]:
maximos_referencia = (
    periodo_referencia[periodo_referencia['element'] == 'TMAX']
    .groupby('month_day')['temperature_c']
    .max()
)

minimos_referencia = (
    periodo_referencia[periodo_referencia['element'] == 'TMIN']
    .groupby('month_day')['temperature_c']
    .min()
)

print('Cantidad de días con récord máximo:', len(maximos_referencia))
print('Cantidad de días con récord mínimo:', len(minimos_referencia))

## 3. Detectar récords superados en 2015
Comparamos cada observación de 2015 con el récord del mismo mes y día calculado para 2005–2014. Marcamos los casos en los que la máxima de 2015 fue mayor que el récord máximo anterior o la mínima de 2015 fue menor que el récord mínimo anterior.

In [ ]:
maximos_2015 = datos_2015[datos_2015['element'] == 'TMAX'].copy()
maximos_2015['record_2005_2014'] = maximos_2015['month_day'].map(maximos_referencia)
record_maximo_superado = maximos_2015[
    maximos_2015['temperature_c'] > maximos_2015['record_2005_2014']
]

minimos_2015 = datos_2015[datos_2015['element'] == 'TMIN'].copy()
minimos_2015['record_2005_2014'] = minimos_2015['month_day'].map(minimos_referencia)
record_minimo_superado = minimos_2015[
    minimos_2015['temperature_c'] < minimos_2015['record_2005_2014']
]

print('Récords máximos superados en 2015:', len(record_maximo_superado))
print('Récords mínimos superados en 2015:', len(record_minimo_superado))

display(record_maximo_superado[['date', 'temperature_c', 'record_2005_2014']].head())
display(record_minimo_superado[['date', 'temperature_c', 'record_2005_2014']].head())

## 4. Visualizar los récords y los valores de 2015
El gráfico muestra las máximas y mínimas récord de 2005–2014 como líneas, sombrea el área entre ambas y agrega puntos de 2015 únicamente cuando se superó alguno de esos récords. El eje horizontal representa el día del año (sin 29 de febrero).

In [ ]:
# Ordenamos los días para que las líneas se dibujen cronológicamente.
dias = sorted(set(maximos_referencia.index) & set(minimos_referencia.index))
maximos_plot = maximos_referencia.reindex(dias)
minimos_plot = minimos_referencia.reindex(dias)

fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(dias, maximos_plot.values, color='firebrick', linewidth=1.2,
        label='Récord máximo diario (2005–2014)')
ax.plot(dias, minimos_plot.values, color='royalblue', linewidth=1.2,
        label='Récord mínimo diario (2005–2014)')
ax.fill_between(dias, minimos_plot.values, maximos_plot.values,
                color='lightgray', alpha=0.35, label='Rango entre récords')

# Convertimos las fechas de los récords superados a la misma clave mes-día.
dias_max_2015 = record_maximo_superado['date'].dt.strftime('%m-%d')
dias_min_2015 = record_minimo_superado['date'].dt.strftime('%m-%d')
ax.scatter(dias_max_2015, record_maximo_superado['temperature_c'],
           color='darkorange', s=24, zorder=3, label='Máxima de 2015 que supera récord')
ax.scatter(dias_min_2015, record_minimo_superado['temperature_c'],
           color='deepskyblue', s=24, zorder=3, label='Mínima de 2015 que supera récord')

# Etiquetas mensuales para facilitar la lectura del eje horizontal.
ticks_meses = [f'{mes:02d}-01' for mes in range(1, 13)]
etiquetas_meses = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun',
                   'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
ax.set_xticks(ticks_meses)
ax.set_xticklabels(etiquetas_meses)
ax.set_title('Récords diarios de temperatura (2005–2014) y récords superados en 2015')
ax.set_xlabel('Mes del año')
ax.set_ylabel('Temperatura (°C)')
ax.legend(loc='best', frameon=False, ncol=2)
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
plt.show()

## 5. Conclusión
El gráfico permite comparar los extremos de temperatura de cada día del año durante 2005–2014 con las observaciones de 2015. Los puntos naranjas indican días en los que la temperatura máxima de 2015 superó el récord máximo previo; los puntos celestes indican días en los que la temperatura mínima de 2015 fue inferior al récord mínimo previo. La cantidad exacta de récords superados se imprime en la sección 3 al ejecutar el notebook.